In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DecimalType
from pyspark.sql.functions import trim, col, to_date

# Read from bronze table

In [0]:
df = spark.table("bike_data.bronze.sales_details_raw")

In [0]:
df.display()

In [0]:
df.printSchema()

# Transformation

## Trimming

In [0]:
for item in df.schema.fields:
    if isinstance(item.dataType, StringType):
        df = df.withColumn(item.name, trim(col(item.name)))

## Normalization

In [0]:
# datatype conversion
# intgers to date using helper function
def clean_date(df, col_name):
    col_str = col(col_name).cast("string")
    return df.withColumn(
        col_name,
        F.when((col_str == "0") | (F.length(col_str) != 8), None)
        .otherwise(to_date(col_str, "yyyyMMdd"))
    )

df = clean_date(df, "sls_order_dt")
df = clean_date(df, "sls_ship_dt")
df = clean_date(df, "sls_due_dt")

In [0]:
df.printSchema()

In [0]:
# datatype conversion
# intgers to decimal for monetary calculations
df = (
    df.withColumn("sls_sales", col("sls_sales").cast(DecimalType(10,2)))
    .withColumn("sls_price", col("sls_price").cast(DecimalType(10,2)))
)

In [0]:
df.display()

In [0]:
df.printSchema()

## Rename Columns

In [0]:
renamed_cols = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_col, new_col in renamed_cols.items():
    df = df.withColumnRenamed(old_col, new_col)

In [0]:
df.limit(5).display()

# Write to  Silver table

In [0]:
(
    df.write
    .mode("overwrite")
    .saveAsTable("bike_data.silver.crm_sales")
)